In [49]:
import numpy as np
import pandas as pd
import os

In [50]:
NUM_ROWS = 7500
OUTPUT_DIR = "data"
OUTPUT_FILE = "pricing_data.csv"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [51]:
#multipliers
df = pd.DataFrame({
    'duration_hours':      np.random.randint(1, 6, size=NUM_ROWS),
    'lead_time_hours':     np.random.uniform(0.5, 168, size=NUM_ROWS),
    'zone_center':         np.random.choice([0, 1], size=NUM_ROWS),
    'severity_score':      np.random.randint(0, 4, size=NUM_ROWS),
    'gender_pref':         np.random.choice([0, 1], size=NUM_ROWS), 
    'min_rating_required': np.random.choice([0, 4.0, 4.5, 5.0], size=NUM_ROWS, p=[0.5, 0.25, 0.2, 0.05]),
    'demand_supply_ratio': np.random.uniform(0.5, 3.0, size=NUM_ROWS)
})

In [52]:
services = ["General House Cleaning",
      "Home Organizing",
      "Deep Cleaning",
      "Aircond Cleaning",
      "Carpet Cleaning",
      "Post-Renovation Cleaning",
      "Sofa or Mattress Cleaning",  
      "Plumbing Services",
      "Air Conditioner Repair",
      "Electrical Repair",
      "Washing Machine Repair",
      "Refrigerator Repair",
      "Door & Lock Repair",
      "Ceiling Repair", 
      "Furniture Assembly",
      "Mounting",
      "Painting & Touch-up Work",
      "Curtain or Blind Installation",
      "Minor Welding Jobs",
      "Kitchen Remodeling",
      "Tiling & Flooring",
      "Electrical Safety Check",
      "Gas Leak Detection",
      "Fire Extinguisher Servicing", 
      "House Moving",
      "Large Item Delivery",
      "Small Item Delivery", 
      "Lawn Mowing",
      "Gardening",
      "Tree Cutting",
      "Roof or Gutter Cleaning"
    ]

#base rate per hour or per unit
base_rates = {
    'general_house_cleaning': 24,
    'home_organizing': 22,
    'deep_cleaning': 32,
    'aircond_cleaning': 40,
    'carpet_cleaning': 30,
    'post_renovation_cleaning': 60,
    'sofa_or_mattress_cleaning': 36,
    'plumbing_services': 60,
    'air_conditioner_repair': 58,
    'electrical_repair': 60,
    'washing_machine_repair': 54,
    'refrigerator_repair': 45,
    'door_lock_repair': 38,
    'ceiling_repair': 44,
    'furniture_assembly': 24,
    'mounting': 34,
    'painting_touch_up_work': 42,
    'curtain_or_blind_installation': 28,
    'minor_welding_jobs': 60,
    'kitchen_remodeling': 85,
    'tiling_flooring': 72,
    'electrical_safety_check': 38,
    'gas_leak_detection': 42,
    'fire_extinguisher_servicing': 38,
    'house_moving': 44,
    'large_item_delivery': 36,
    'small_item_delivery': 20,
    'lawn_mowing': 34,
    'gardening': 28,
    'tree_cutting': 48,
    'roof_or_gutter_cleaning': 52
}

service_choices = np.random.choice(services, size=NUM_ROWS)
df['service'] = service_choices
dummies = pd.get_dummies(df['service'], prefix='service')

# Replace all spaces to underscore and lowercase all letters
dummies.columns = (
    dummies.columns
           .str.lower()
           .str.replace(r'[^0-9a-z_]+','_', regex=True)
           .str.rstrip('_')
)

# One-hot encoding
df = pd.concat([df, dummies], axis=1)

# Compute base rate
def get_base_rate(row):
    for col in dummies.columns:
        if col in row and row[col] == 1:
            key = col.replace('service_', '')
            return base_rates.get(key, 0)
    return 0

df['base_rate'] = df.apply(get_base_rate, axis=1)

def duration_price(base_rate, duration):
    if duration <= 1:
        return base_rate
    elif duration == 2:
        return base_rate + base_rate * np.random.uniform(0.6, 0.9)

    elif duration == 3:
        return base_rate + base_rate * np.random.uniform(0.5, 0.8)

    else:
        return base_rate + duration_price(base_rate, duration - 1)

def simulate_order_placed(row):
    price = row['price']
    base = row['base_rate'] 
    duration = row['duration_hours']
    base_price = duration_price(base, duration)

    if base * 0.7 <= price <= base_price * 1.8:
        return np.random.choice([1, 0], p=[0.85, 0.15])
    elif price <= base_price * 0.7:
        return np.random.choice([1, 0], p=[0.75, 0.25])
    elif price <= base_price * 3:
        return np.random.choice([1, 0], p=[0.5, 0.5])
    else:
        return np.random.choice([1, 0], p=[0.5, 0.5])

def simulate_task_accepted(row):
    price = row['price']
    base = row['base_rate'] 
    duration = row['duration_hours']
    base_price = duration_price(base, duration)

    if row['order_placed'] == 0:
        return 0

    # If urgency high + low price, less likely to accept
    if row['lead_time_hours'] <= 12 and price < base_price * 1.1:
        return np.random.choice([1, 0], p=[0.5, 0.5])
    
    # If high severity but low pay
    if row['severity_score'] >= 2 and row['price'] < base_price * 1.2:
        return np.random.choice([1, 0], p=[0.35, 0.65])

    # Generally good cases
    return np.random.choice([1, 0], p=[0.9, 0.1])

def simulate_task_completed(row):
    if row['task_accepted'] == 1:
        return np.random.choice([1, 0], p=[0.95, 0.05])
    return 0
    
# Price calculation
def final_price(row):
    price = duration_price(row['base_rate'], row['duration_hours'])

    # Urgency multiplier
    if row['lead_time_hours'] <= 24:
        price *= np.random.uniform(1.17, 1.2)
    elif row['lead_time_hours'] <= 72:
        price *= np.random.uniform(1.05, 1.1)

    # Zone multiplier
    price *= np.random.uniform(1.05, 1.08) if row['zone_center'] == 1 else 1.0

    # Gender preferences
    if row['gender_pref'] == 1:
        price *= np.random.uniform(1.02, 1.05)

    # Rating multiplier
    if row['min_rating_required'] >= 4.5:
        price *= np.random.uniform(1.05, 1.1)
    if row['min_rating_required'] >= 4.8:
        price *= np.random.uniform(1.1, 1.15)
    if row['min_rating_required'] == 5.0:
        price *= np.random.uniform(1.15, 1.2)

    # Severity multiplier
    price *= 1 + row['severity_score'] * np.random.uniform(0.3, 0.33)

    # Surge pricing with cap
    surge_multiplier = min(max(row['demand_supply_ratio'], 0.8), 1.1)
    price *= np.random.uniform(surge_multiplier - 0.03, surge_multiplier + 0.03)
    # noise = np.random.normal(loc=0, scale=price * 0.05)
    price = price 

    # # Avoid having a lower price than the base rate
    # min_price = base_duration_price
    # max_price = base_duration_price * 1.8
    
    # price = max(min_price, price)
    # price = min(max_price, price)

    return round(price, 2)

df['price'] = df.apply(final_price, axis=1)
df['order_placed'] = df.apply(simulate_order_placed, axis=1)
df['task_accepted'] = df.apply(simulate_task_accepted, axis=1)
df['task_completed'] = df.apply(simulate_task_completed, axis=1)


In [52]:
services = ["General House Cleaning",
      "Home Organizing",
      "Deep Cleaning",
      "Aircond Cleaning",
      "Carpet Cleaning",
      "Post-Renovation Cleaning",
      "Sofa or Mattress Cleaning",  
      "Plumbing Services",
      "Air Conditioner Repair",
      "Electrical Repair",
      "Washing Machine Repair",
      "Refrigerator Repair",
      "Door & Lock Repair",
      "Ceiling Repair", 
      "Furniture Assembly",
      "Mounting",
      "Painting & Touch-up Work",
      "Curtain or Blind Installation",
      "Minor Welding Jobs",
      "Kitchen Remodeling",
      "Tiling & Flooring",
      "Electrical Safety Check",
      "Gas Leak Detection",
      "Fire Extinguisher Servicing", 
      "House Moving",
      "Large Item Delivery",
      "Small Item Delivery", 
      "Lawn Mowing",
      "Gardening",
      "Tree Cutting",
      "Roof or Gutter Cleaning"
    ]

#base rate per hour or per unit
base_rates = {
    'general_house_cleaning': 24,
    'home_organizing': 22,
    'deep_cleaning': 32,
    'aircond_cleaning': 40,
    'carpet_cleaning': 30,
    'post_renovation_cleaning': 60,
    'sofa_or_mattress_cleaning': 36,
    'plumbing_services': 60,
    'air_conditioner_repair': 58,
    'electrical_repair': 60,
    'washing_machine_repair': 54,
    'refrigerator_repair': 45,
    'door_lock_repair': 38,
    'ceiling_repair': 44,
    'furniture_assembly': 24,
    'mounting': 34,
    'painting_touch_up_work': 42,
    'curtain_or_blind_installation': 28,
    'minor_welding_jobs': 60,
    'kitchen_remodeling': 85,
    'tiling_flooring': 72,
    'electrical_safety_check': 38,
    'gas_leak_detection': 42,
    'fire_extinguisher_servicing': 38,
    'house_moving': 44,
    'large_item_delivery': 36,
    'small_item_delivery': 20,
    'lawn_mowing': 34,
    'gardening': 28,
    'tree_cutting': 48,
    'roof_or_gutter_cleaning': 52
}

service_choices = np.random.choice(services, size=NUM_ROWS)
df['service'] = service_choices
dummies = pd.get_dummies(df['service'], prefix='service')

# Replace all spaces to underscore and lowercase all letters
dummies.columns = (
    dummies.columns
           .str.lower()
           .str.replace(r'[^0-9a-z_]+','_', regex=True)
           .str.rstrip('_')
)

# One-hot encoding
df = pd.concat([df, dummies], axis=1)

# Compute base rate
def get_base_rate(row):
    for col in dummies.columns:
        if col in row and row[col] == 1:
            key = col.replace('service_', '')
            return base_rates.get(key, 0)
    return 0

df['base_rate'] = df.apply(get_base_rate, axis=1)

def duration_price(base_rate, duration):
    if duration <= 1:
        return base_rate
    elif duration == 2:
        return base_rate + base_rate * np.random.uniform(0.6, 0.9)

    elif duration == 3:
        return base_rate + base_rate * np.random.uniform(0.5, 0.8)

    else:
        return base_rate + duration_price(base_rate, duration - 1)

def simulate_order_placed(row):
    price = row['price']
    base = row['base_rate'] 
    duration = row['duration_hours']
    base_price = duration_price(base, duration)

    if base * 0.7 <= price <= base_price * 1.8:
        return np.random.choice([1, 0], p=[0.85, 0.15])
    elif price <= base_price * 0.7:
        return np.random.choice([1, 0], p=[0.75, 0.25])
    elif price <= base_price * 3:
        return np.random.choice([1, 0], p=[0.5, 0.5])
    else:
        return np.random.choice([1, 0], p=[0.5, 0.5])

def simulate_task_accepted(row):
    price = row['price']
    base = row['base_rate'] 
    duration = row['duration_hours']
    base_price = duration_price(base, duration)

    if row['order_placed'] == 0:
        return 0

    # If urgency high + low price, less likely to accept
    if row['lead_time_hours'] <= 12 and price < base_price * 1.1:
        return np.random.choice([1, 0], p=[0.5, 0.5])
    
    # If high severity but low pay
    if row['severity_score'] >= 2 and row['price'] < base_price * 1.2:
        return np.random.choice([1, 0], p=[0.35, 0.65])

    # Generally good cases
    return np.random.choice([1, 0], p=[0.9, 0.1])

def simulate_task_completed(row):
    if row['task_accepted'] == 1:
        return np.random.choice([1, 0], p=[0.95, 0.05])
    return 0
    
# Price calculation
def final_price(row):
    price = duration_price(row['base_rate'], row['duration_hours'])

    # Urgency multiplier
    if row['lead_time_hours'] <= 24:
        price *= np.random.uniform(1.17, 1.2)
    elif row['lead_time_hours'] <= 72:
        price *= np.random.uniform(1.05, 1.1)

    # Zone multiplier
    price *= np.random.uniform(1.05, 1.08) if row['zone_center'] == 1 else 1.0

    # Gender preferences
    if row['gender_pref'] == 1:
        price *= np.random.uniform(1.02, 1.05)

    # Rating multiplier
    if row['min_rating_required'] >= 4.5:
        price *= np.random.uniform(1.05, 1.1)
    if row['min_rating_required'] >= 4.8:
        price *= np.random.uniform(1.1, 1.15)
    if row['min_rating_required'] == 5.0:
        price *= np.random.uniform(1.15, 1.2)

    # Severity multiplier
    price *= 1 + row['severity_score'] * np.random.uniform(0.3, 0.33)

    # Surge pricing with cap
    surge_multiplier = min(max(row['demand_supply_ratio'], 0.8), 1.1)
    price *= np.random.uniform(surge_multiplier - 0.03, surge_multiplier + 0.03)
    # noise = np.random.normal(loc=0, scale=price * 0.05)
    price = price 

    # # Avoid having a lower price than the base rate
    # min_price = base_duration_price
    # max_price = base_duration_price * 1.8
    
    # price = max(min_price, price)
    # price = min(max_price, price)

    return round(price, 2)

df['price'] = df.apply(final_price, axis=1)
df['order_placed'] = df.apply(simulate_order_placed, axis=1)
df['task_accepted'] = df.apply(simulate_task_accepted, axis=1)
df['task_completed'] = df.apply(simulate_task_completed, axis=1)


In [53]:
# Save and preview
output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILE)
df.to_csv(output_path, index=False)
print(f"Pricing data saved to: {output_path}")
print(df[['service', 'base_rate', 'price', 'order_placed', 'task_accepted', 'task_completed']].head())


Pricing data saved to: data/pricing_data.csv
                     service  base_rate   price  order_placed  task_accepted  \
0             Ceiling Repair         44   44.90             1              1   
1     Washing Machine Repair         54  195.39             1              1   
2  Sofa or Mattress Cleaning         36   85.57             1              1   
3         Minor Welding Jobs         60  169.92             1              0   
4        Refrigerator Repair         45   95.18             1              1   

   task_completed  
0               1  
1               1  
2               1  
3               0  
4               1  


In [54]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7500 entries, 0 to 7499
Data columns (total 44 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   duration_hours                         7500 non-null   int64  
 1   lead_time_hours                        7500 non-null   float64
 2   zone_center                            7500 non-null   int64  
 3   severity_score                         7500 non-null   int64  
 4   gender_pref                            7500 non-null   int64  
 5   min_rating_required                    7500 non-null   float64
 6   demand_supply_ratio                    7500 non-null   float64
 7   service                                7500 non-null   object 
 8   service_air_conditioner_repair         7500 non-null   bool   
 9   service_aircond_cleaning               7500 non-null   bool   
 10  service_carpet_cleaning                7500 non-null   bool   
 11  serv